In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 0 — Install deps + Mount Drive
# ════════════════════════════════════════════════════════════
!pip install -q transformers datasets scikit-learn tqdm pandas accelerate pyarrow

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Config
# ════════════════════════════════════════════════════════════
import torch
import transformers
transformers.logging.set_verbosity_error()

CHUNK_SIZE          = 512
STRIDE              = 256
MAX_CHUNKS          = 16
GRU_HIDDEN          = 256
GRU_LAYERS          = 2
DROPOUT             = 0.3
BERT_LR             = 2e-5        # low LR for unfrozen BERT layers
GRU_LR              = 3e-4        # higher LR for GRU + classifier
WEIGHT_DECAY        = 0.01
BATCH_SIZE          = 4
GRAD_ACCUM_STEPS    = 4           # effective batch = 4 * 4 = 16
EPOCHS              = 15
PATIENCE            = 3
WARMUP_RATIO        = 0.1
UNFREEZE_LAYERS     = 3           # unfreeze last 3 BERT encoder layers
MODEL_NAME          = "law-ai/InLegalBERT"
SAVE_PATH           = "/content/drive/MyDrive/CAPSTONE/inlegalbertgru/best_hierarchical_model.pt"
DEVICE              = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Tokenizer + Dataset
# ════════════════════════════════════════════════════════════
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ILDCHierarchicalDataset(Dataset):

    def __init__(self, df, tokenizer, chunk_size=512, stride=256, max_chunks=16):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.chunk_size = chunk_size
        self.stride     = stride
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.df)

    def _chunk_text(self, text):
        encoding  = self.tokenizer(str(text), add_special_tokens=False, return_tensors=None)
        all_ids   = encoding['input_ids']
        all_mask  = encoding['attention_mask']
        effective = self.chunk_size - 2
        chunks, start = [], 0

        while start < len(all_ids):
            end        = start + effective
            chunk_ids  = [tokenizer.cls_token_id] + all_ids[start:end]  + [tokenizer.sep_token_id]
            chunk_mask = [1]                       + all_mask[start:end] + [1]
            pad        = self.chunk_size - len(chunk_ids)
            chunk_ids  += [tokenizer.pad_token_id] * pad
            chunk_mask += [0] * pad
            chunks.append({
                'input_ids':      torch.tensor(chunk_ids,  dtype=torch.long),
                'attention_mask': torch.tensor(chunk_mask, dtype=torch.long)
            })
            if end >= len(all_ids):
                break
            start += self.stride

        return chunks[:self.max_chunks]

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        chunks = self._chunk_text(row['text'])
        label  = torch.tensor(int(row['label']), dtype=torch.long)
        return chunks, label


def collate_fn(batch):
    chunk_lists, labels = zip(*batch)
    max_chunks = max(len(c) for c in chunk_lists)
    chunk_size = chunk_lists[0][0]['input_ids'].shape[0]
    batch_ids, batch_masks, chunk_counts = [], [], []

    for chunks in chunk_lists:
        n     = len(chunks)
        chunk_counts.append(n)
        ids   = torch.stack([c['input_ids']     for c in chunks])
        masks = torch.stack([c['attention_mask'] for c in chunks])
        pad   = max_chunks - n
        if pad > 0:
            ids   = torch.cat([ids,   torch.zeros(pad, chunk_size, dtype=torch.long)])
            masks = torch.cat([masks, torch.zeros(pad, chunk_size, dtype=torch.long)])
        batch_ids.append(ids)
        batch_masks.append(masks)

    return {
        'input_ids':      torch.stack(batch_ids),
        'attention_mask': torch.stack(batch_masks),
        'chunk_counts':   torch.tensor(chunk_counts, dtype=torch.long),
        'labels':         torch.tensor(labels,       dtype=torch.long)
    }


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Model
# ════════════════════════════════════════════════════════════
import torch.nn as nn
from transformers import AutoModel


class HierarchicalLegalBERT(nn.Module):

    def __init__(self, model_name, gru_hidden=256, gru_layers=2,
                 dropout=0.3, num_classes=2, unfreeze_last_n=3):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        # Freeze all BERT params first
        for param in self.bert.parameters():
            param.requires_grad = False

        # Unfreeze last N encoder layers + pooler
        num_layers = len(self.bert.encoder.layer)
        for i in range(num_layers - unfreeze_last_n, num_layers):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = True
        for param in self.bert.pooler.parameters():
            param.requires_grad = True

        bert_dim = self.bert.config.hidden_size  # 768

        self.gru = nn.GRU(
            input_size    = bert_dim,
            hidden_size   = gru_hidden,
            num_layers    = gru_layers,
            batch_first   = True,
            dropout       = dropout if gru_layers > 1 else 0,
            bidirectional = True
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(gru_hidden * 2, num_classes)

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Trainable params : {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

    def forward(self, input_ids, attention_mask, chunk_counts):
        B, max_chunks, seq_len = input_ids.shape
        ids   = input_ids.view(B * max_chunks, seq_len)
        masks = attention_mask.view(B * max_chunks, seq_len)

        # Gradient flows through unfrozen BERT layers
        bert_out = self.bert(input_ids=ids, attention_mask=masks)
        cls_emb  = bert_out.last_hidden_state[:, 0, :]   # (B*max_chunks, 768)
        cls_emb  = cls_emb.view(B, max_chunks, -1)
        cls_emb  = self.dropout(cls_emb)

        packed = nn.utils.rnn.pack_padded_sequence(
            cls_emb, chunk_counts.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)

        # Concat final forward + backward hidden states
        final = torch.cat([hidden[-2], hidden[-1]], dim=1)
        final = self.dropout(final)
        return self.classifier(final)


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Optimizer + Scheduler
# ════════════════════════════════════════════════════════════
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


def build_optimizer_and_scheduler(model, num_training_steps):
    num_layers     = len(model.bert.encoder.layer)
    bert_param_ids = set()

    for i in range(num_layers - UNFREEZE_LAYERS, num_layers):
        for p in model.bert.encoder.layer[i].parameters():
            bert_param_ids.add(id(p))
    for p in model.bert.pooler.parameters():
        bert_param_ids.add(id(p))

    bert_params  = [p for p in model.parameters() if id(p) in bert_param_ids  and p.requires_grad]
    other_params = [p for p in model.parameters() if id(p) not in bert_param_ids and p.requires_grad]

    optimizer = AdamW([
        {'params': bert_params,  'lr': BERT_LR, 'weight_decay': WEIGHT_DECAY},
        {'params': other_params, 'lr': GRU_LR,  'weight_decay': WEIGHT_DECAY},
    ])

    warmup_steps = int(WARMUP_RATIO * num_training_steps)
    scheduler    = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = num_training_steps
    )
    print(f"Total steps: {num_training_steps}  |  Warmup steps: {warmup_steps}")
    return optimizer, scheduler

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Train / Eval loops
# ════════════════════════════════════════════════════════════
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

scaler = torch.cuda.amp.GradScaler()


def train_epoch(model, loader, optimizer, scheduler, criterion, device, accum_steps=4):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="  Train")):
        ids    = batch['input_ids'].to(device)
        masks  = batch['attention_mask'].to(device)
        counts = batch['chunk_counts']
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast():
            logits = model(ids, masks, counts)
            loss   = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss  += loss.item() * accum_steps
        all_preds   += logits.argmax(dim=-1).detach().cpu().tolist()
        all_labels  += labels.cpu().tolist()

    return {
        'loss': total_loss / len(loader),
        'acc':  accuracy_score(all_labels, all_preds),
        'f1':   f1_score(all_labels, all_preds, average='macro')
    }


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc="  Eval "):
        ids    = batch['input_ids'].to(device)
        masks  = batch['attention_mask'].to(device)
        counts = batch['chunk_counts']
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast():
            logits = model(ids, masks, counts)
            loss   = criterion(logits, labels)

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=-1).cpu().tolist()
        all_labels  += labels.cpu().tolist()

    return {
        'loss': total_loss / len(loader),
        'acc':  accuracy_score(all_labels, all_preds),
        'f1':   f1_score(all_labels, all_preds, average='macro')
    }


/tmp/ipykernel_2431/2716975639.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Load preprocessed data from Drive
# ════════════════════════════════════════════════════════════
import pandas as pd


def load_data():
    train_path = '/content/drive/MyDrive/CAPSTONE/Data/preprocessed_ildc/criminal_train.parquet'
    dev_path   = '/content/drive/MyDrive/CAPSTONE/Data/preprocessed_ildc/criminal_dev.parquet'

    print("Loading preprocessed data from Drive...")
    train_df = pd.read_parquet(train_path)
    dev_df   = pd.read_parquet(dev_path)

    print(f"Train : {len(train_df):,}  |  columns: {train_df.columns.tolist()}")
    print(f"Dev   : {len(dev_df):,}  |  columns: {dev_df.columns.tolist()}")

    for df, name in [(train_df, 'train'), (dev_df, 'dev')]:
        assert 'text'  in df.columns, f"'{name}' missing 'text' column — found: {df.columns.tolist()}"
        assert 'label' in df.columns, f"'{name}' missing 'label' column — found: {df.columns.tolist()}"

    print(f"\nTrain — Label 0: {(train_df['label']==0).sum():,} | Label 1: {(train_df['label']==1).sum():,}")
    print(f"Dev   — Label 0: {(dev_df['label']==0).sum():,}   | Label 1: {(dev_df['label']==1).sum():,}")

    return train_df, dev_df


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Main run()
# ════════════════════════════════════════════════════════════
import os

def run():
    # ── Load data ────────────────────────────────────────────
    train_df, dev_df = load_data()

    train_ds = ILDCHierarchicalDataset(train_df, tokenizer, CHUNK_SIZE, STRIDE, MAX_CHUNKS)
    dev_ds   = ILDCHierarchicalDataset(dev_df,   tokenizer, CHUNK_SIZE, STRIDE, MAX_CHUNKS)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    # ── Model ────────────────────────────────────────────────
    model = HierarchicalLegalBERT(
        MODEL_NAME, GRU_HIDDEN, GRU_LAYERS, DROPOUT,
        unfreeze_last_n=UNFREEZE_LAYERS
    ).to(DEVICE)

    # Memory check
    torch.cuda.empty_cache()
    print(f"GPU memory after model load — "
          f"allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB | "
          f"reserved: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")

    # ── Weighted loss for class imbalance ────────────────────
    label_counts  = train_df['label'].value_counts().sort_index().values
    class_weights = torch.tensor([1.0/label_counts[0], 1.0/label_counts[1]], dtype=torch.float)
    class_weights = class_weights / class_weights.sum()
    criterion     = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    # ── Optimizer + scheduler ────────────────────────────────
    total_steps          = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
    optimizer, scheduler = build_optimizer_and_scheduler(model, total_steps)

    # ── Make sure save directory exists ─────────────────────
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

    # ── Training loop ────────────────────────────────────────
    best_f1      = 0.0
    patience_ctr = 0
    history      = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\n{'='*55}")
        print(f"  Epoch {epoch}/{EPOCHS}")
        print(f"{'='*55}")

        train_metrics = train_epoch(model, train_loader, optimizer, scheduler,
                                    criterion, DEVICE, GRAD_ACCUM_STEPS)
        dev_metrics   = eval_epoch(model, dev_loader, criterion, DEVICE)

        print(f"  Train — loss: {train_metrics['loss']:.4f} | "
              f"acc: {train_metrics['acc']:.4f} | macro-F1: {train_metrics['f1']:.4f}")
        print(f"  Dev   — loss: {dev_metrics['loss']:.4f} | "
              f"acc: {dev_metrics['acc']:.4f} | macro-F1: {dev_metrics['f1']:.4f}")

        history.append({'epoch': epoch, 'train': train_metrics, 'dev': dev_metrics})

        if dev_metrics['f1'] > best_f1:
            best_f1      = dev_metrics['f1']
            patience_ctr = 0
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1':              best_f1,
                'config': {
                    'model_name':      MODEL_NAME,
                    'chunk_size':      CHUNK_SIZE,
                    'stride':          STRIDE,
                    'max_chunks':      MAX_CHUNKS,
                    'gru_hidden':      GRU_HIDDEN,
                    'gru_layers':      GRU_LAYERS,
                    'unfreeze_layers': UNFREEZE_LAYERS,
                }
            }, SAVE_PATH)
            print(f"  ✓ Best model saved  (dev macro-F1: {best_f1:.4f})")
        else:
            patience_ctr += 1
            print(f"  No improvement. Patience: {patience_ctr}/{PATIENCE}")
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    print(f"\nDone. Best dev macro-F1: {best_f1:.4f}")
    return history


In [9]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Run
# ════════════════════════════════════════════════════════════
history = run()

Loading preprocessed data from Drive...
Train : 7,094  |  columns: ['id', 'text', 'label']
Dev   : 154  |  columns: ['id', 'text', 'label']

Train — Label 0: 4,095 | Label 1: 2,999
Dev   — Label 0: 77   | Label 1: 77


pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

Trainable params : 24,613,890 / 112,241,922 (21.9%)
GPU memory after model load — allocated: 0.45 GB | reserved: 0.49 GB
Total steps: 6645  |  Warmup steps: 664

  Epoch 1/15


  Train:   0%|          | 0/1774 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval :   0%|          | 0/39 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval : 100%|██████████| 39/39 [00:19<00:00,  2.02it/s]


... (9 outputs left out)

  Train:   0%|          | 0/1774 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval :   0%|          | 0/39 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval : 100%|██████████| 39/39 [00:19<00:00,  2.00it/s]


  Train — loss: 0.4245 | acc: 0.8096 | macro-F1: 0.8058
  Dev   — loss: 0.6833 | acc: 0.7013 | macro-F1: 0.7008
  No improvement. Patience: 2/3

  Epoch 7/15


  Train:   0%|          | 0/1774 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval :   0%|          | 0/39 [00:00<?, ?it/s]/tmp/ipykernel_2431/2716975639.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval : 100%|██████████| 39/39 [00:19<00:00,  2.00it/s]

  Train — loss: 0.3735 | acc: 0.8362 | macro-F1: 0.8331
  Dev   — loss: 0.7683 | acc: 0.7013 | macro-F1: 0.7011
  No improvement. Patience: 3/3

  Early stopping at epoch 7.

Done. Best dev macro-F1: 0.7319
